# 학교 PDF 데이터 수집 (카메라 0)
4.Dofbot_Data수집_라즈베리파이5.pdf 12~17쪽 화면의 코드를 옮겼습니다.
카메라 번호만 1에서 0으로 변경했습니다. 설명용 Markdown과 셀 구분을 추가했습니다.

PC에 게임패드를 연결하고 위에서부터 실행합니다. **첫 코드 셀부터 팔이 [92, 77, 11, 18, 99, 91]로 이동합니다.**
주변을 확인한 뒤 실행하세요. Run All로 재실행하면 스레드가 중복될 수 있으므로, 재수집 전 Kernel Restart를 사용합니다.
SELECT(8)는 초기 자세 이동, 버튼 9는 수집 종료입니다. CSV의 각도는 목표각이며 실제 측정 각도가 아닙니다.
현재 video0이 촬영 가능한 장치인지 먼저 확인해야 합니다. 아래 셀은 아직 실행하지 않았습니다.


In [ ]:
import ipywidgets.widgets as widgets
import threading
import time
import cv2
import os
import datetime
import csv
from Arm_Lib import Arm_Device

Arm = Arm_Device()
controller = widgets.Controller(index=0)
display(controller)

is_recording = True
last_button_state = 0
current_session_path = ""
target_angles = [92, 77, 11, 18, 99, 91]
a1, a2, a3, a4, a5, a6 = target_angles

def move_to_target():
    """로봇을 지정된 초기 위치로 이동시키는 함수"""
    global a1, a2, a3, a4, a5, a6
    print(f"초기 위치로 이동 중: {target_angles}")
    Arm.Arm_serial_servo_write6(*target_angles, 2000)
    a1, a2, a3, a4, a5, a6 = target_angles
    time.sleep(2.5)

move_to_target()


## 세션 생성 및 조이스틱 제어 함수 (PDF 13~14쪽)

In [ ]:
session_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
current_session_path = f"dataset_{session_time}"
os.makedirs(os.path.join(current_session_path, "cam1"), exist_ok=True)

csv_path = os.path.join(current_session_path, "labels.csv")
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['frame_id', 'timestamp', 'm1', 'm2', 'm3', 'm4', 'm5', 'm6'])

print(f"프로그램 시작: 단일 카메라 데이터 수집 진행 중: {current_session_path}")

def Arm_Handle():
    global is_recording, last_button_state, current_session_path
    global a1, a2, a3, a4, a5, a6
    s_time, s_step = 800, 1
    print("준비 완료. '9'번 버튼을 누르면 데이터 수집 및 제어를 완전히 종료합니다.")
    while is_recording:
        if len(controller.axes) < 6 or len(controller.buttons) < 10:
            time.sleep(0.5)
            continue
        current_button_state = controller.buttons[9].value
        if current_button_state == 1 and last_button_state == 0:
            is_recording = False
            print(f"9번 버튼 입력됨: 수집을 종료합니다: {current_session_path}")
            break
        last_button_state = current_button_state
        try:
            time.sleep(0.05)
            if abs(controller.axes[1].value) > 0.1:
                a2 = max(0, min(180, a2 + (s_step if controller.axes[1].value > 0.1 else -s_step)))
                Arm.Arm_serial_servo_write(2, a2, s_time)
            if abs(controller.axes[0].value) > 0.1:
                a1 = max(0, min(180, a1 - (s_step if controller.axes[0].value > 0.1 else -s_step)))
                Arm.Arm_serial_servo_write(1, a1, s_time)
            if controller.buttons[1].value:
                a5 = min(180, a5 + s_step); Arm.Arm_serial_servo_write(5, a5, s_time)
            elif controller.buttons[3].value:
                a5 = max(0, a5 - s_step); Arm.Arm_serial_servo_write(5, a5, s_time)
            if controller.buttons[0].value: a6 = min(180, a6 + s_step)
            elif controller.buttons[2].value: a6 = max(0, a6 - s_step)
            if controller.buttons[0].value or controller.buttons[2].value: Arm.Arm_serial_servo_write(6, a6, s_time)
            if controller.buttons[5].value: a4 = max(0, a4 - s_step)
            elif controller.buttons[7].value: a4 = min(180, a4 + s_step)
            if controller.buttons[5].value or controller.buttons[7].value: Arm.Arm_serial_servo_write(4, a4, s_time)
            if controller.buttons[4].value: a3 = max(0, a3 - s_step)
            elif controller.buttons[6].value: a3 = min(180, a3 + s_step)
            if controller.buttons[4].value or controller.buttons[6].value: Arm.Arm_serial_servo_write(3, a3, s_time)
            if controller.buttons[8].value: move_to_target()
            time.sleep(0.1)
        except Exception as e:
            print(f"제어 에러: {e}"); break


## 데이터 수집 함수 (PDF 15쪽, 카메라 번호 0으로 변경)

In [ ]:
def Data_Collection():
    global is_recording, current_session_path, a1, a2, a3, a4, a5, a6
    cap1 = cv2.VideoCapture(0)
    cap1.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    interval = 1.0 / 3.0
    frame_count = 0
    try:
        while is_recording:
            start_time = time.time()
            ret1, frame1 = cap1.read()
            if ret1:
                frame_count += 1
                img1 = cv2.resize(frame1, (244, 244))
                img_name = f"{frame_count:05d}.jpg"
                ts = datetime.datetime.now().strftime('%H%M%S.%f')[:-3]
                csv_path = os.path.join(current_session_path, "labels.csv")
                with open(csv_path, 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([frame_count, ts, a1, a2, a3, a4, a5, a6])
                cv2.imwrite(os.path.join(current_session_path, "cam1", img_name),
                            img1, [cv2.IMWRITE_JPEG_QUALITY, 85])
            elapsed = time.time() - start_time
            wait = interval - elapsed
            if wait > 0:
                time.sleep(wait)
            else:
                time.sleep(0.001)
    finally:
        cap1.release()


## 제어·수집 스레드 시작 (PDF 17쪽)
버튼 9로 종료합니다. 재시작 전 커널을 재시작하세요.

In [ ]:
t_ctrl = threading.Thread(target=Arm_Handle, daemon=True)
t_data = threading.Thread(target=Data_Collection, daemon=True)
t_ctrl.start()
t_data.start()
